In [1]:
from pathlib import Path
import sys
sys.path.insert(0, Path( "../..").resolve().absolute().__str__() )

For working with a list of lists, it is natural to deal with indexes to guide through it. This can be useful, as it allows to use math to identify the right folders, rather than through word selection in dictionaries.

The math required involves three key elements:
- **Folder** or **File ID**: Represents the target folder as a list of indexes
    - I.e. [0, 1, 2] = "Open 1st folder, then the second subfolder inside it, then the 3rd subsubfolder inside it. (using 0 indexing)
- **Folders Count**: Maps / represents the folder system of the hierarquical tree as a list
    - I.e. [ [2], [0, 2], [0, 0 ]] = "1st Level (root dir) has 2 subfolder, 2nd Level has 0 subsubfolders in 1st subfolder & 2 subsubfolders in 2nd subfolder, 3rd Level has 0 subsubsubfolders in each subsubfolder.
- **File Tree**: This is the nested list of list, which the last two objects will be used to guide which list to open
    

### <ins> Quick Example Of Operating </ins>

- #### Example of Data Structures - Folder Counts | File Tree

In [12]:
from tests.testFileSys import testTree
from xaidar.treeObj import createTree, viewSubtree, visualizeTree

filetree, folderTree, treeDepth, _ , foldersCount, __ = createTree( testTree )
folderID = [0]

print("\nFolders Count Structure")
for idx, count in enumerate( foldersCount ): print( f"L{idx + 1}: {count}")
print("\nFile Tree -> Nested List of Lists Structure")
visualizeTree( filetree)
print("\nFile Tree -> File System Representation")
viewSubtree(filetree, foldersCount, 1, treeDepth + 2, folderID = folderID)



Folders Count Structure
L1: [1]
L2: [2]
L3: [3, 2]
L4: [0, 1, 1, 0, 1]
L5: [0, 1, 0]
L6: [0]

File Tree -> Nested List of Lists Structure
  1	| [['root']]
  2	| [['subDir11', 'subDir12', 'file11.txt']]
  3	| [['subDir21', 'subDir22', 'subDir23', 'file21.txt'], ['subDir24', 'subDir25', 'file22.txt']]
  4	| [['file31.txt', 'file32.txt', 'file33.txt'], ['subDir31', 'file34.txt'], ['subDir32', 'file35.txt'], ['file36.txt', 'file37.txt'], ['subDir33', 'file38.txt']]
  5	| [['file41.txt', 'file42.txt', 'file43.txt'], ['subDir41', 'file44.txt', 'file45.txt'], ['file46.txt', 'file47.txt']]
  6	| [['file51.txt']]
  7	| []

File Tree -> File System Representation
root
├── [0] subDir11
│   ├── [0] subDir21
│   │   ├── [0-f] file31.txt
│   │   ├── [1-f] file32.txt
│   │   └── [2-f] file33.txt
│   ├── [1] subDir22
│   │   ├── [0] subDir31
│   │   │   ├── [0-f] file41.txt
│   │   │   ├── [1-f] file42.txt
│   │   │   └── [2-f] file43.txt
│   │   └── [1-f] file34.txt
│   ├── [2] subDir23
│   │   ├── 

- #### Example and Interpredation of - Folder ID

By looking at the <ins>File Tree - File System Representation</ins>, let's pick a file such as **file38.txt**.

Now, to get the **fileID**, we go down the file system and open folders along the way:
- Open **root** -> fileID = [0]
- Then open **subDir12** -> fileID = [0, 1]
- Then open **subDir25** -> fileID = [0, 1, 1]
- Finally we open **file38.txt** -> fileID = [0, 1, 1, 1]

However, say we want to open **subDir25** (where file38.txt lives) in the **tree** object. 

The way to go about it is to iteratively open the last element of the nested list of lists, until we reach the **subDir25** Level, and then open the list index corresponding to the **subDir25** folder.

To do this, we need the **subDir25** folder ID, which is the fileID without the last element (which is pointing at the file within the folder). 
<p align="center"> FolderID = [0, 1, 1] </p>

In [8]:
print("  Level 1: ",filetree[:-1])
print("  Level 2: ",filetree[-1][:-1])
print("  Level 3: ",filetree[-1][-1][:-1])
print("\nLevel with target file38.txt: ")
print("  Level 4: ",filetree[-1][-1][-1][:-1])

  Level 1:  [['root']]
  Level 2:  [['subDir11', 'subDir12', 'file11.txt']]
  Level 3:  [['subDir21', 'subDir22', 'subDir23', 'file21.txt'], ['subDir24', 'subDir25', 'file22.txt']]

Level with target file38.txt: 
  Level 4:  [['file31.txt', 'file32.txt', 'file33.txt'], ['subDir31', 'file34.txt'], ['subDir32', 'file35.txt'], ['file36.txt', 'file37.txt'], ['subDir33', 'file38.txt']]


- #### Example and Calculation of - Gamma


So, when we open the **tree object** at the Level where the target file lives, we get a list with lists, where each list represents a folder, and the content of the list represents the folder content.

I.e. 

Folders -> &emsp;&emsp;&emsp;  [&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;subDir11&emsp;&emsp;&emsp;&emsp;,&emsp;&emsp;&emsp;&emsp;&emsp;&emsp; subDir12 &emsp;&emsp;&emsp;&emsp;&emsp;]

Tree -> [ ['subDir21', 'subDir22', 'subDir23', 'file21.txt'], ['subDir24', 'subDir25', 'file22.txt'] ]

Thus, to get the content of our target folder we need to know what is the right index for that folder. Thus, we need to convert our **FolderID** into an **Index** which we'll name **Gamma**.

<p align = "center">Folder ID -> Gamma Index</p>

This is where the key maths comes along.

To do this we need the: &emsp; **foldersCount** & **folderID**.

The key idea is that the same way that we follow which subfolder to open, and which subsubfolder to open with the subfolder, we need to do the same with indexes.

So, we do it naturally where in information about the previous folder guides us to the next subfolder. In this case, it will be the information of the previous **Gamma**.


Take: 

&emsp;&emsp; FolderID = [ 0, 1, 1 ] 

&emsp;&emsp; foldersCount = [ [1], [2], [3, 2], [0, 1, 1, 0, 1], [0, 1, 0], [0] ]




<details>

<summary> Overview 

<p></p>

**Gamma Formula** 

$$

Level \space 1\space Gamma = 0\newline

Level \space n \space Gamma = \\ sum( foldercount_{Level_{n-1}}[:Gamma_{Level_{n-1}}]) + FolderID_{Level_{n-1}}

$$



**I.e.** 

&emsp; Folder ID = [0, 1, 1] & Folder Count = [ [1], [2], [3, 2], ...]

&emsp; Folder Path = "root/subDir12/subDir25"

- Level 1 Gamma = 0
- Level 2 Gamma = sum( Level1_foldercount[ : 0]) + Level1_FolderID = sum( [ ] ) + 0 = 0
- Level 3 Gamma = sum( Level2_foldercount[ : 0]) + Level2_FolderID = sum( [ ] ) + 1 = 1
- Level 4 Gamma = sum( Level3_foldercount[ : 1]) + Level3_FolderID = sum( [3] ) + 1 = 4


tree[-1][-1][-1][4] -> Open folder with folderID [0, 1, 1] -> ['subDir33', 'file38.txt'] 
</summary>

<ins>**Level 1:**</ins>

Gamma = 0 -> At level one, gamma is always zero, because there is only one list - which represents the "supraroot" - in the tree at this level that contains all the folders and files for that level. From the next level onwards, there can be one of 

Tree -> [['root']]

Folder Count [ [1], ...]-> shows that there is only one folder in Level 2.(aka 1 subfolder in Level 1). Aka inside the root folder there is only one subfolder.

FolderID [ 0, ...] -> shows that we must select the 1st subfolder in level 1. 

Folder Path -> "root"

<ins> **Level 2:**</ins>

**Gamma = 0**  -> So, the target folder in Level 2 will be the 1st Folder (), because there is only one folder in level 2, there is only one folder in Level 1, and we want to open the first subfolder of that folder.

Tree -> [['subDir11', 'subDir12', 'file11.txt']]

Folder Count [ [1], [2], ...] -> shows that there are two folders in Level 3 (aka 2 subfolders in Level 2)

FolderID [ 0, 1, ... ] -> shows that we target the must select the 2nd subfolder 

**Folder Path** -> "root/subdir12"

<ins> **Level 3:**</ins>

**Gamma = 1**  -> Look at Level 3 Tree


**Tree** -> [ ['subDir21', 'subDir22', 'subDir23', 'file21.txt'], ['subDir24', 'subDir25', 'file22.txt'] ]

**Folder Count** [ [1], [2], [3, 2], ...] -> shows that there are five folders (3+2) in Level 4 (aka 3 subfolders from first folder in Level 2 and 2 subfolders from 2nd folder in Level 2).

This means that we'll have to pick to

**FolderID**= [ 0, 1, 1 ]  -> shows that we target the must select the 1st subfolder in the 2nd folder. (aka subdir24 inside subdir12 )

**FolderPath** -> "root/subdir12/subdir24"


<ins> **Level 4:**</ins>

**Gamma = 3**  -> 3 + 1 = 4 ( remember zero indexing )
- Given that the target folder lives inside the 2nd suprafolder, when looking at the tree in level 4, we will have to count the first three folders inside the 1st suprafolder, and only then get the one on the second suprafolder. 


Tree -> [ ['file31.txt', 'file32.txt', 'file33.txt'], ['subDir31', 'file34.txt'], ['subDir32', 'file35.txt'], ['file36.txt', 'file37.txt'], ['subDir33', 'file38.txt'] ]


Note:
- Each element in folder count says how many folders are in the next level
- Each element in folderID says which folder to open in the next level - or which subfolder it is targetting in the current level

</details>




In Practice:

In [ ]:
from xaidar.treeObj import createTree, getGamma, pinchLevel
filetree, folderTree, treeDepth, _ , foldersCount, __ = createTree( testTree )
gamma = getGamma( [ [1], [2], [3, 2]], [0,1,1] )
print(f"Calculated Gamma: ", gamma )

folderContent = pinchLevel( filetree, len( [0,1,1] ) + 1 )[gamma]

print(f"Items found within folder with ID [0,1,1]: {folderContent}")

Calculated Gamma:  4
Items found within folder with ID [0,1,1]: ['subDir33', 'file38.txt']


In [3]:
from pathlib import Path
import sys
sys.path.insert(0, Path( "../..").resolve().absolute().__str__() )

from tests.testFileSys import testTree
from xaidar.treeObj import sortPaths, createFolderTree,countFolders, viewSubtree, visualizeTree


orderedPaths = sortPaths( testTree)

folderTree, treeDepth, _ = createFolderTree( orderedPaths )
foldersCount = countFolders( folderTree, treeDepth  )
folderID = [0]
viewSubtree(folderTree, foldersCount, 1, treeDepth + 2, folderID = folderID)
visualizeTree( folderTree)
print("\n",foldersCount)

root
├── [0] subDir11
│   ├── [0] subDir21
│   ├── [1] subDir22
│   │   └── [0] subDir31
│   └── [2] subDir23
│       └── [0] subDir32
│           └── [0] subDir41
└── [1] subDir12
    ├── [0] subDir24
    └── [1] subDir25
        └── [0] subDir33
  1	| [['root']]
  2	| [['subDir11', 'subDir12']]
  3	| [['subDir21', 'subDir22', 'subDir23'], ['subDir24', 'subDir25']]
  4	| [[], ['subDir31'], ['subDir32'], [], ['subDir33']]
  5	| [[], ['subDir41'], []]
  6	| [[]]
  7	| []

 [[1], [2], [3, 2], [0, 1, 1, 0, 1], [0, 1, 0], [0]]


In [10]:
lst = [1,[]]
print(id(lst) )
lst2 = lst
print( id(lst2))
print(id(lst))
lst2 = lst2[-1]
print( id(lst2), lst2)
print(id(lst), lst)
lst2 = lst
print( id(lst2), lst2)
print(id(lst), lst)

3253357307648
3253357307648
3253357307648
3253357299584 []
3253357307648 [1, []]
3253357307648 [1, []]
3253357307648 [1, []]
